# $$\text{Loading and Visualizing the PZT-data}$$

The general structure of the dataset is

```text
Data/
├── Layup1/
│   ├── Documentation/
│   ├── Specimens: L1_S11_F, L1_S12_F, L1_S18_F, L1_S19_F
│   └── README.pdf
│
├── Layup2/
│   ├── Documentation/
│   ├── Specimens: L2_S11_F, L2_S17_F, L2_S18_F, L2_S20_F
│   └── README.pdf
│
└── Layup3/
    ├── Documentation/
    ├── Specimens: L3_S11_F, L3_S13_F, L3_S14_F, L3_S18_F, L3_S20_F
    └── README.pdf

Each specimen folder:
├── PZT-data/
├── StrainData/
├── XRays/
└── L{i}S{j}.xlsx
```

**Important!** Since several entries do not contain strain data, it will not be used in any current prediction. Also, if the training goes as planned I will include a way for the Artificial Neural Network (ANN) to validate its own prediction up against the measured ground truth (the x-ray images).

Another important distinction is the way the data has been measured. They've measured the samples in three different states,

1. Measured loaded in the test machine
2. Clamped (not loaded) in the test machine
3. Traction free (removed from the test machine)

They also measured twice for most of the measurements. Thus, I want to simplify it by only using the first traction free measurement.

# $$\text{Preliminaries}$$

In [1]:
import numpy as np
import scipy.io
import re

from pathlib import Path
from scipy.io import loadmat

### $$\text{Finding the file-paths}$$

There are some inconsistencies in the dataset. Such as the PZT-data being named *PZTdata* for "Data/Layup3/LS_S13_F". The PZT-data is also missing from "L3_S14_F".

In [2]:
root = Path("Data")

specimens = {
    "Layup1": ["L1_S11_F", "L1_S12_F", "L1_S18_F", "L1_S19_F"],
    "Layup2": ["L2_S11_F", "L2_S17_F", "L2_S18_F", "L2_S20_F"],
    "Layup3": ["L3_S11_F", "L3_S13_F", "L3_S18_F", "L3_S20_F"]  # L3_S14_F does not contain any PZT-data
}

In [3]:
def get_cycle_number(path):
    """
    Extract cycle number from filenames like:
    L1S11_100000_3_1.mat
    """
    match = re.search(r"_(\d+)_3_1\.mat$", path.name)

    if match is None:
        raise ValueError(f"Could not extract cycle number from {path.name}")

    return int(match.group(1))

In [4]:
PZT_files = {}

for layup, specimen_list in specimens.items():
    PZT_files[layup] = {}

    for specimen in specimen_list:
        pzt_path = root / layup / specimen / "PZT-data"

        files = sorted(
            pzt_path.glob("*_3_1.mat"),
            key=get_cycle_number
        )

        PZT_files[layup][specimen] = files

In [5]:
PZT_files           # Quick sanity check

{'Layup1': {'L1_S11_F': [PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_0_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_1_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_10_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_100_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_1000_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_10000_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_20000_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_30000_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_40000_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_50000_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_60000_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_70000_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_80000_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_90000_3_1.mat'),
   PosixPath('Data/Layup1/L1_S11_F/PZT-data/L1S11_1000

## $$\text{1 - Loading the data}$$

In [6]:
def mat_scalar(value, default=None):
    value = np.asarray(value).squeeze()

    if value.size == 0:
        return default

    return value.item()


In [7]:
def mat_string(value):
    value = mat_scalar(value, default="")

    if value is None:
        return ""

    return str(value)

In [8]:
def load_pzt_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    coupon = mat["coupon"]

    paths = np.atleast_1d(coupon.path_data)

    path_meta = [
        {
            "actuator": int(mat_scalar(p.actuator)),
            "sensor": int(mat_scalar(p.sensor)),
            "frequency": int(mat_scalar(p.frequency)),
            "amplitude": int(mat_scalar(p.amplitude)),
            "gain": int(mat_scalar(p.gain)),
            "sampling_rate": int(mat_scalar(p.sampling_rate)),
        }
        for p in paths
    ]

    return {
        "file": path,
        "cycles": int(mat_scalar(coupon.cycles)),
        "load": mat_scalar(coupon.load, default=None),
        "condition": mat_string(coupon.condition).lower(),
        "comment": mat_string(coupon.comment),
        "path_meta": path_meta,
        "signal_actuator": np.stack([p.signal_actuator for p in paths]),
        "signal_sensor": np.stack([p.signal_sensor for p in paths]),
    }

In [9]:
PZT_data = {
    layup: {
        specimen: [load_pzt_mat(path) for path in files]
        for specimen, files in specimen_dict.items()
    }
    for layup, specimen_dict in PZT_files.items()
}

This returns

```python

sample = PZT_data["Layup1"]["L1_S111_F"][0]

sample.keys()

===> dict_keys(['file', 'cycles', 'load', 'condition', 'comment', 
                'path_meta', 'signal_actuator', 'signal_sensor'])

```


## $$\text{2 - Control}$$

In [10]:
signal_sensor_data = {}

for layup, specimen_dict in PZT_data.items():
    signal_sensor_data[layup] = {}

    for specimen, measurements in specimen_dict.items():
        signal_sensor_data[layup][specimen] = {}

        for sample in measurements:
            cycle = sample["cycles"]

            signal_sensor_data[layup][specimen][cycle] = sample["signal_sensor"]

In [ ]:
len(signal_sensor_dat

3